# N036 · 扫描线与事件排序

**目标：** 用事件增减维护时间或空间中的活跃对象。

**先修：** N034, N035。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 起止事件；同坐标次序；离线处理；最大重叠；面积扩展。

## 从问题到算法

扫描线把区间变成开始与结束事件，活动数的最大值就是最大重叠。闭区间在同一坐标应先加开始、再减结束；半开区间则相反。端点顺序不是实现细节，而是区间语义的一部分。将区间分到互不重叠组时，最大重叠给出必要下界，按时间释放旧组可达到该下界。

## 最小实现

**本章接口：** `max_overlap(intervals, closed)`、`min_groups(intervals)`

In [1]:
def max_overlap(intervals,closed=True):
    events=[]
    for left,right in intervals:
        if left>right: raise ValueError('reversed interval')
        if not closed and left==right: continue
        events.append((left,1)); events.append((right,-1))
    events.sort(key=lambda event:(event[0],-event[1] if closed else event[1]))
    active=best=0
    for _,delta in events:
        active+=delta; best=max(best,active)
    return best

def min_groups(intervals):
    return max_overlap(intervals,closed=True)

## 正确性、前提与复杂度

活动数在相邻事件间不变，只需检查事件位置的合法中间状态。达到最大重叠的那一刻所有区间必须分组不同；按开始时间复用最早空出的组不会超过同时活动数。

**代价：** O(n log n) 排序时间、O(n) 事件空间。

## 小实例与可视化

先手算，再运行。下表由当前内核中的代码和显式小实例生成；不是预制答案图片。

In [2]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

a=[[1,2],[2,3],[2,2]]
show_table(['区间语义','最大重叠'],[('闭区间',max_overlap(a,True)),('半开区间',max_overlap(a,False))])
events=sorted([(x,d) for l,r in a for x,d in [(l,1),(r,-1)]],key=lambda z:(z[0],-z[1]))
rows=[]; active=0
for x,d in events:
    active+=d; rows.append((x,d,active))
show_table(['坐标','事件增量','活动区间数'],rows)

区间语义,最大重叠
闭区间,3
半开区间,1


坐标,事件增量,活动区间数
1,1,1
2,1,2
2,1,3
2,-1,2
2,-1,1
3,-1,0


## 自动测试

以下断言检查实现契约。测试通过不等同于一般性证明，也不表示已经提交 LeetCode 官方评测。

In [3]:
assert max_overlap([[1,2],[2,3]],True)==2
assert max_overlap([[1,2],[2,3]],False)==1
assert min_groups([[1,10],[2,9],[3,8]])==3
assert max_overlap([],True)==0
from random import Random
rng=Random(36)
for _ in range(200):
    a=[sorted([rng.randrange(5),rng.randrange(5)]) for _ in range(5)]
    points=[i/2 for i in range(9)]
    assert max_overlap(a,True)==max(sum(l<=x<=r for l,r in a) for x in points)
    assert max_overlap(a,False)==max(sum(l<=x<r for l,r in a) for x in points)
print('N036: 所有本章断言通过')

N036: 所有本章断言通过


## 练习：先独立完成

**练习 1：** 分别实现闭区间和半开区间扫描。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

**练习 2：** 说明同坐标事件顺序改变答案。写明输入输出与边界，先给一个手算示例，再用本章接口或独立暴力解验证。

未完成练习不阻断教学区 Run All。验收时解释为什么正确，不只展示运行截图。

## 代表题与迁移

- 1094. Car Pooling（canonical）
- 2406. Divide Intervals Into Minimum Number of Groups（canonical）

这些题用于知识映射；本章实现遵循上文明确的教学契约。除原规格注明已核对身份的条目外，本轮未重新联网核验全部题面、收费权限或平台语言支持。不要把同概念教学实例当作所有官方题的完整答案。

## 自测与下一步

不看代码，复述状态、不变量和一个失效条件；改变一个输入约束，判断实现是否仍成立。确认能解释边界测试后，按 `specs/curriculum.json` 的 `learning_order` 进入下一章。